# Physics Lab 3/4: TPX3
This is the notebook used for the TPX3 experiment. It contains all of the code needed to convert, process, and plot the measurement data from the detector, excluding calibration: this code you will have to write yourself. 

The measurement files are reasonably large. To save you from having to upload the files and wait hours every time you work on this, you can mount a shared Google Drive folder containing the measurements. Alternatively, you can download all of the files and work on them locally.

This document was last updated on: 04-01-2022.

## Included functions
These are all of the functions used to process the measurement data, as well as the libraries that they depend on. You can keep this minimised and run the entire cell block or look more into the functions to get a better idea of what they do and even modify them as you see fit.

In [ ]:
import os
import numpy as np
import time
import matplotlib.pyplot as plt

tic = time.time()

def calculate_energy(tottime, cal_val, lin=False):
    """
    Function that uses given values for calibration to convert from ToT to energy.
    """
    a = cal_val[0]
    b = cal_val[1]
    c = cal_val[2]
    t = cal_val[3]
    y = tottime
    if not lin:
        # print('use non-lin')
        if a == 0:
            print("Error: a = 0")
            return 0
        elif (
            np.sqrt(
                a * a * t * t
                + 2 * a * b * t
                + 4 * a * c
                - 2 * a * t * y
                + b * b
                - 2 * b * y
                + y * y
            )
            + y
            == a * t + b
        ):
            print("Error: second condition not ok")
            return 0
        else:
            # If borh conditions are met, calculate energy from tot
            return float(
                (
                    np.sqrt(
                        a * a * t * t
                        + 2 * a * b * t
                        + 4 * a * c
                        - 2 * a * t * y
                        + b * b
                        - 2 * b * y
                        + y * y
                    )
                    + a * t
                    - b
                    + y
                )
                / (2 * a)
            )
    else:
        # use linear
        if a == 0:
            print("Error: a = 0")
            return 0
        else:
            return float((y - b) / a)


def calculate_tottime(energy, cal_val, lin=False):
    """
    Function that uses given values for calibration to convert from energy to ToT.
    """
    a = cal_val[0]
    b = cal_val[1]
    c = cal_val[2]
    t = cal_val[3]
    if not lin:
        if energy == t:
            print("Error: division by zero")
            return 0
        elif energy <= (
            (np.sqrt(a * a * t * t + 2 * a * b * t + 4 * a * c + b * b) + a * t - b)
            / (2 * a)
        ):
            return 0
        elif (
            -1 * np.sqrt(a * a * t * t + 2 * a * b * t + 4 * a * c + b * b) + a * t + b
            == 0
        ):
            return 0
        else:
            return float(a * energy + b - c / (energy - t))
    else:
        return a * energy + b


def plot_calibration_curve(cal_val, lin=False):
    """
    Function that uses given values for calibration to plot a calibration curve, showing the relation between ToT and energy.
    """
    print("Saving calibration curve")
    xval = range(200)
    yval = []
    for val in xval:
        yval.append(calculate_tottime(val, cal_val, lin=lin))
    fig = plt.figure()
    plt.plot(xval, yval)
    plt.title("Calibration curve")
    plt.xlabel("E")
    plt.ylabel("ToT")
    plt.text(
        0.2,
        0.6,
        f"a = {cal_val[0]}\nb= {cal_val[1]}\nc= {cal_val[2]}\nt= {cal_val[3]}\nlin= {lin}",
        transform=fig.transFigure,
    )
    plt.savefig("calibrationcurve.png")
    plt.close()


def is_integer_num(n):
    """
    Function that checks whether an integer can be made from input n.
    """
    try:
        integer = int(n)
    except:
        return False
    else:
        return True






def convert_t3pa(filename):
    """
    Function that converts the t3pa file to a .npy. Requires the pixelCoordinates variable.
    """
    tic = time.time()
    # Adding a 'maximum' amount of hits to prevent a google colab crash
    memory_limit = 50000000
    
    # Dictionary with pixel coordinates
    # Convert matrix index to (x,y) pixel coordinates
    pixelCoordinates = dict()
    i = 0
    for y in range(0, 256):
        for x in range(0, 256):
            pixelCoordinates[i] = [x, y]
            i += 1
    
    try:
        with open(f"./{filename}.t3pa", "r") as file:
            print(f"Processing results for file: {filename}.t3pa")
            print(f"Convert to .npy file")
            line_count = 0
            for line in file:
                if line_count > memory_limit:
                  break
                else:
                  line_count += 1
            pixdat = np.zeros((line_count, 5))
            
        print(f"Processing {int(line_count):8d} hits")
        
        with open(f"./{filename}.t3pa", "r") as file:
            line_index = 0
            
            for line in file:
                # Limit in hits to be processed so google colab doesn't crash
                if line_index > memory_limit:
                  print("Processing stopped to preserve RAM")
                  break
                
                dataline = line.split()
                # ['Index', 'MatrixIndex', 'ToA', 'ToT', 'FToA', 'Overflow']
                #  0        1              2      3      4       5
                if line_index > 1 and line_index % 10000000 == 0:
                    # Calculate how long the processing is going to take
                    toc = time.time()
                    elapsed_time = (toc - tic) / 60  # min
                    time_remaining = float(
                        100 * elapsed_time / (line_index / line_count * 100)
                        - elapsed_time
                    )  # min
                    time_finished = toc + 60 * time_remaining
                    print(
                        f"{int(line_index / line_count * 100):3d}% - {time_remaining:3.2f} minutes remaining"
                        f" - finished at {time.ctime(time_finished)}"
                    )
                    
                if is_integer_num(dataline[0]):
                    # Processing the data.
                    index = int(dataline[0])

                    # matrix to position
                    position = pixelCoordinates[int(dataline[1])]
                    # toa[ns] = toa        * 25 - ftoa        * 25/16
                    toatime = float(int(dataline[2]) * 25 - int(dataline[4]) * 25 / 16)
                    tottime = int(dataline[3])
                    xpos = position[0]
                    ypos = position[1]

                    pixdat[line_index, 0] = index
                    pixdat[line_index, 1] = xpos
                    pixdat[line_index, 2] = ypos
                    pixdat[line_index, 3] = toatime
                    pixdat[line_index, 4] = tottime
                    line_index += 1

        # remove empty lines
        pixdat = pixdat[~np.all(pixdat == 0, axis=1)]
        # oder array on ToA_time
        pixdat = pixdat[pixdat[:, 3].argsort()]
        # save pixdat to .npy
        np.save(f"./{filename}/{filename}_pixeldata", pixdat)
        print(f"File saved as {filename}_pixeldata.npy in folder {filename}")

        toc = time.time()
        dt = time.gmtime(toc - tic)
        print(f'Converting took {time.strftime("%H:%M:%S", dt)} seconds')

    except IOError:
        print(f"file not found ({filename}.t3pa)")


def cluster_finder(filename, time_window, pixel_range, verbose_level=1):
    """
    Function that finds clusters in the measurement file by looking at events that occur at neighbouring pixels within a given time window.
    Input variables:
    filename: filename
    time_window: maximum time between pixel 'hits'
    pixel_range: maximum range in which neighbouring pixels are detected.
    verbose_level:
    """
    tic = time.time()
    try:
        # open input file with pixeldata
        pixdat = np.load(f"./{filename}/{filename}_pixeldata.npy", "r")
        print(f"Looking for Clusters in {filename}_pixeldata.npy")
        print(f"Window: {time_window}ns\nPixel range:{pixel_range:.2f}")
        # add extra column to matrix for cluster_index
        cluster_matrix = np.zeros((len(pixdat), 6))
        cluster_matrix[:, :-1] = pixdat

        cluster_index_counter = 0
        total_pix = len(pixdat)

        for i in range(total_pix):
            if verbose_level > 1:
                print(i)
            if i > 1 and i % 1000000 == 0:
                # calculate time remaining
                toc = time.time()
                elapsed_time = (toc - tic) / 60  # min
                time_remaining = float(
                    100 * elapsed_time / (i / total_pix * 100) - elapsed_time
                )  # min
                time_finished = toc + 60 * time_remaining
                print(
                    f"{int(i/ total_pix *100):3d}% - {time_remaining:3.2f} minutes remaining"
                    f" - finished at {time.ctime(time_finished)}"
                )

            if int(cluster_matrix[i, 5]) == 0:
                # print(f'pixel i = {i} is not used')
                cluster_index_counter += 1
                cluster_matrix[i, 5] = cluster_index_counter
                index_current_cluster = [i]
                counter = 0

                while len(index_current_cluster) > counter:
                    current_index = index_current_cluster[counter]
                    t_min = cluster_matrix[current_index, 3]

                    for j in range(current_index + 1, len(cluster_matrix)):
                        if (
                            i != j or cluster_matrix[j, 5] == 0
                        ):  # same pixel or already considered
                            time_diff = cluster_matrix[j, 3] - t_min
                            if verbose_level > 2:
                                print("else2")
                                print(cluster_matrix[j, 2])
                                print(time_diff)
                                print(time_window)
                                
                            # stop searching some time after...
                            if time_diff >= time_window:
                                if verbose_level > 2:
                                    print("break")
                                    print("max time, break")
                                break
                                
                            if abs(time_diff) < time_window:
                                if verbose_level > 1:
                                    print("check dist")

                                # check for distance
                                distance = np.sqrt(
                                    (
                                        cluster_matrix[current_index, 1]
                                        - cluster_matrix[j, 1]
                                    )
                                    * (
                                        cluster_matrix[current_index, 1]
                                        - cluster_matrix[j, 1]
                                    )
                                    + (
                                        cluster_matrix[current_index, 2]
                                        - cluster_matrix[j, 2]
                                    )
                                    * (
                                        cluster_matrix[current_index, 2]
                                        - cluster_matrix[j, 2]
                                    )
                                )
                                if abs(distance) <= pixel_range:
                                    if verbose_level > 1:
                                        print("dist2")
                                        print(
                                            f"possible cluster for {int(pixdat[current_index, 0]):d} "
                                            f"({int(pixdat[current_index, 1]):d}, {int(pixdat[current_index, 2]):d}) "
                                            f"and {int(pixdat[j, 0]):d} ({int(pixdat[j, 1]):d}, {int(pixdat[j, 2]):d})"
                                        )
                                    cluster_matrix[j, 5] = cluster_index_counter
                    counter += 1
        # sort matrix on cluster_index
        cluster_matrix = cluster_matrix[cluster_matrix[:, 5].argsort()]
        np.save(f"./{filename}/{filename}_clusters.npy", cluster_matrix)

        toc = time.time()
        dt = time.gmtime(toc - tic)
        print(f'Clustering took {time.strftime("%H:%M:%S",dt)}')
    except Exception as e:
        print(e)


def cluster_npy_text(filename, cal_val, lin=False):
    """
    Function that saves the cluster data to a .txt file.
    """
    try:
        print("save cluster information to text file")
        tic = time.time()

        clusterdata = np.load(f"./{filename}/{filename}_clusters.npy")
        with open(f"./{filename}/{filename}_clusterstat.txt", "w+") as f_cluster:
            with open(f"./{filename}/{filename}_clusterdata.txt", "w+") as f:
                f.write("Clusterdata\n\n")
                f.write(
                    "index    xPos    yPos    ToATime         ToTTime        ToTEnergy\n"
                )
                cluster_index = 1
                max_cluster_size = 0
                ndx = 0

                e_tot = 0.0
                t_tot = 0.0
                x_tot = 0.0
                y_tot = 0.0
                times = []
                cluster_size = 0

                # new clustermatrix
                # index, clustersize, tot, e (res)
                cluster_matrix = np.zeros((len(clusterdata), 4))

                for i in range(len(clusterdata)):
                    if int(cluster_index) == int(clusterdata[i, 5]):
                        # print(f"pix {i} belongs to cluster {cluster_index}")
                        e = calculate_energy(clusterdata[i, 4], cal_val, lin=lin)
                        f.write(
                            f"{int(clusterdata[i, 0]):5d} {int(clusterdata[i, 1]):7d} "
                            f"{int(clusterdata[i, 2]):7d} {int(clusterdata[i, 3]):14.4f} "
                            f"{int(clusterdata[i, 4]):8d} {e:8f}\n"
                        )
                        # add values for calculation
                        e_tot += e
                        t_tot += clusterdata[i, 4]
                        x_tot += clusterdata[i, 1]
                        y_tot += clusterdata[i, 2]
                        times += [clusterdata[i, 3]]
                        cluster_size += 1
                    else:
                        # save cluster so far
                        x_avg = x_tot / cluster_size
                        y_avg = y_tot / cluster_size
                        if cluster_size > max_cluster_size:
                            max_cluster_size = cluster_size
                        # calculate time difference (window)
                        t_min = min(times)
                        t_max = max(times)
                        d_t = t_max - t_min
                        f.write(
                            f"({x_avg:.1f}, {y_avg:.1f}), E = {e_tot:.1f}, Ttot = {t_tot:.1f}, "
                            f"clustersize = {cluster_size:d}, time window = {d_t:.1f}\n\n"
                        )

                        #                     0                  1             2
                        #                   size               energy         tot
                        f_cluster.write(
                            f"{cluster_size:04d}  {e_tot:08.3f}  {t_tot:08.3f}\n"
                        )

                        cluster_matrix[ndx, :] = [ndx, cluster_size, t_tot, e]
                        ndx += 1

                        # start new cluster
                        e = calculate_energy(clusterdata[i, 4], cal_val, lin=lin)
                        f.write(
                            f"{int(clusterdata[i, 0]):5d} {int(clusterdata[i, 1]):7d} "
                            f"{int(clusterdata[i, 2]):7d} {int(clusterdata[i, 3]):14.4f} "
                            f"{int(clusterdata[i, 4]):8d} {e:8f}\n"
                        )

                        e_tot = e
                        t_tot = clusterdata[i, 4]
                        x_tot = clusterdata[i, 1]
                        y_tot = clusterdata[i, 2]
                        times = [clusterdata[i, 3]]
                        cluster_size = 1

                        cluster_index += 1
                x_avg = x_tot / cluster_size
                y_avg = y_tot / cluster_size
                if cluster_size > max_cluster_size:
                    max_cluster_size = cluster_size
                # calculate time difference (window)
                t_min = min(times)
                t_max = max(times)
                d_t = t_max - t_min
                f.write(
                    f"({x_avg:.1f}, {y_avg:.1f}), E = {e_tot:.1f}, Ttot = {t_tot:.1f}, "
                    f"clustersize = {cluster_size:d}, time window = {d_t:.1f}\n\n"
                )
                f_cluster.write(f"{cluster_size:04d}  {e_tot:08.3f}  {t_tot:08.3f}\n")
                f.write(f"\nMaximum cluster size: {max_cluster_size} pixels.\n")
                cluster_matrix[ndx, :] = [ndx, cluster_size, t_tot, e]
                ndx += 1
                np.save(f"{filename}/{filename}_clustermatrix.npy")

        toc = time.time()
        dt = time.gmtime(toc - tic)
        print(f'converting to text took {time.strftime("%H:%M:%S", dt)}')
    except Exception as e:
        print(e)


def clusterdata_to_npy(filename, cal_val, lin=False, calc_e=False):
    """
    Function that saves the cluster data to a .npy file.
    """
    try:
        print("Save cluster information to npy file")
        tic = time.time()
        clusterdata = np.load(f"./{filename}/{filename}_clusters.npy")
        toc = time.time()
        dt = time.gmtime(toc - tic)
        cluster_index = 1
        ndx = 0

        e_tot = 0.0
        t_tot = 0.0
        cluster_size = 0

        # new clustermatrix
        # index, clustersize, tot, e (res)
        cluster_matrix = np.zeros((len(clusterdata), 4))

        for i in range(len(clusterdata)):
            if i > 1 and i % 1000000 == 0:
                toc = time.time()
                elapsed_time = (toc - tic) / 60  # min
                time_remaining = float(
                    100 * elapsed_time / (i / len(clusterdata) * 100) - elapsed_time
                )  # min
                time_finished = toc + 60 * time_remaining
                print(
                    f"{int(i/ len(clusterdata) *100):3d}% - {time_remaining:3.2f} minutes remaining"
                    f" - finished at {time.ctime(time_finished)}"
                )
            if int(cluster_index) == int(clusterdata[i, 5]):
                if calc_e:
                    e = calculate_energy(clusterdata[i, 4], cal_val, lin=lin)
                    e_tot += e
                # add values for calculation

                t_tot += clusterdata[i, 4]
                cluster_size += 1
            else:
                cluster_matrix[ndx, :] = [ndx, cluster_size, t_tot, e_tot]
                ndx += 1

                # start new cluster
                if calc_e:
                    e = calculate_energy(clusterdata[i, 4], cal_val, lin=lin)
                    e_tot = e
                t_tot = clusterdata[i, 4]
                cluster_size = 1

                cluster_index += 1
        cluster_matrix[ndx, :] = [ndx, cluster_size, t_tot, e_tot]
        ndx += 1

        np.save(f"{filename}/{filename}_clustermatrix.npy", cluster_matrix)
        toc = time.time()
        dt = time.gmtime(toc - tic)
        print(f'Converting to .npy took {time.strftime("%H:%M:%S", dt)}')
    except Exception as err:
        print(err)


def plot_histograms(
    filename, ylim=0, xrange=0, num_box=50, plt_e=False, single=True, plt_cluster=False, plotlog=False, return_data=False
):
    """
    Function that plots either single-pixel or cluster data in a histogram.
    """
    print("Plotting histograms of cluster")
    try:
        cluster_matrix = np.load(f"{filename}/{filename}_clustermatrix.npy")

        # split in cluster and single pixel array
        single_pixel = cluster_matrix[cluster_matrix[:, 1] == 1]
        clusters = cluster_matrix[cluster_matrix[:, 1] > 1]

        for lim in ylim:
            for range in xrange:
                if plt_cluster:
                    # plots for clusters
                    fig = plt.figure(figsize=fig_size)
                    plt.title("Histogram of cluster size")
                    plt.hist(clusters[:, 1], num_box)
                    plt.xlabel("cluster size")
                    plt.ylabel("counts")
                    plt.text(0.2, 0.6, f"{filename}", transform=fig.transFigure)
                    plt.savefig(
                        f"./{filename}/{filename}_hist_clustersize.png", dpi=dpi
                    )
                    plt.close()

                    if plt_e:
                        fig = plt.figure(figsize=fig_size)
                        plt.title("Histogram of cluster energy")
                        plt.hist(clusters[:, 3], num_box)
                        plt.xlabel("cluster energy (keV)")
                        plt.ylabel("counts")
                        if lim != 0:
                            plt.ylim([0, lim])
                        if range != 0:
                            plt.xlim(range)
                        plt.text(0.2, 0.6, f"{filename}", transform=fig.transFigure)
                        plt.savefig(
                            f"./{filename}/{filename}_hist_clusterenergy_{lim}_{range}.png",
                            dpi=dpi,
                        )
                        plt.close()

                    fig = plt.figure(figsize=fig_size)
                    plt.title("Histogram of cluster ToT value")
                    plt.hist(clusters[:, 2], num_box)
                    plt.xlabel("cluster ToT (counts)")
                    plt.ylabel("counts")
                    if lim != 0:
                        plt.ylim([0, lim])
                    if range != 0:
                        plt.xlim(range)
                    plt.text(0.2, 0.6, f"{filename}", transform=fig.transFigure)
                    plt.savefig(
                        f"./{filename}/{filename}_hist_clustertot_{lim}_{range}.png",
                        dpi=dpi,
                    )
                    plt.close()

                if single:
                    # plots for single pixel
                    if plt_e:
                        fig = plt.figure(figsize=fig_size)
                        # plt.title('Histogram of single_pix cluster energy')
                        plt.hist(single_pixel[:, 3], num_box)
                        plt.xlabel("cluster energy (keV)")
                        plt.ylabel("counts")
                        if lim != 0:
                            plt.ylim([0, lim])
                        if range != 0:
                            plt.xlim(range)
                        # plt.text(0.2, 0.6, f'{filename}', transform=fig.transFigure)
                        plt.savefig(
                            f"./{filename}/{filename}_hist_single_pix_energy_{lim}_{range}.png",
                            dpi=dpi,
                        )
                        plt.close()

                    plt.figure(figsize=fig_size)
                    fig, ax = plt.subplots()

                    # plt.title('Histogram of single_pix cluster ToT value')
                    if not plt_e:
                      plt.hist(single_pixel[:, 2], num_box)
                      plt.xlabel("ToT value (count)")
                    plt.ylabel("counts")

                    if lim != 0:
                        plt.ylim([0, lim])
                    if range != 0:
                        # stepsize = int(range[1] / 10)
                        # plt.xticks(np.arange(0, range[1], stepsize))
                        plt.xlim(range)
                    # plt.text(0.2, 0.6, f'{filename}', transform=fig.transFigure)
                    # ax.yaxis.set_label_position("right")
                    # ax.yaxis.tick_right()
                    if plotlog:
                      plt.yscale('log', nonposy='clip')
                    
                    plt.savefig(
                        f"./{filename}/{filename}_hist_single_pix_tot_{lim}_{range}.png",
                        dpi=dpi,
                    )
                    plt.close()

    except Exception as e:
        print("!EXCEPTION!")
        print(e)
    
    if return_data:
      return single_pixel[:,3] if plt_e else single_pixel[:,2]


def plot_cluster(filename, min_clus_size, verbose_level=1, clim=0):
    """
    Function that plots clusters in a grid.
    """
    print("Plotting clusters")
    try:
        clusterdata = np.load(f"./{filename}/{filename}_clusters.npy")
        padding = 5  # pixels around cluster for plotting
        cluster_index = 1
        cluster = []
        for i in range(len(clusterdata)):
            if int(cluster_index) == int(clusterdata[i, 5]):
                # append to current cluster
                cluster.append(
                    [clusterdata[i, 1], clusterdata[i, 2], clusterdata[i, 4]]
                )
            else:
                # plot current cluster if size > min clus size
                if len(cluster) >= min_clus_size:
                    if verbose_level > 1:
                        print(cluster)
                    # find x_min, x_max, y_min, y_max
                    x_min = int(min(x[0] for x in cluster))
                    x_max = int(max(x[0] for x in cluster))
                    y_min = int(min(y[1] for y in cluster))
                    y_max = int(max(y[1] for y in cluster))
                    if verbose_level > 1:
                        print(cluster)
                        print(x_min)
                        print(x_max)
                        print(y_min)
                        print(y_max)
                        
                    grid = np.zeros((256, 256))
                    for pix in cluster:
                        grid[int(pix[1]), int(pix[0])] = pix[2]
                        if verbose_level > 1:
                            print(f"x={int(pix[0])}, y={int(pix[1])}, ToT={pix[2]}")
                            
                    t = float(clusterdata[i, 3])
                    fig = plt.figure(figsize=[10, 10], dpi=600)
                    ax = fig.add_subplot(111)
                    plt.imshow(grid)
                    plt.colorbar()
                    if clim != 0:
                        plt.clim(0, clim)
                        
                    if verbose_level > 1:
                        print(x_min - padding)
                        print(x_max + padding)
                        print(y_min - padding)
                        print(y_max + padding)
                        
                    plt.xlim(x_min - padding, x_max + padding)
                    plt.ylim(y_min - padding, y_max + padding)
                    props = ""
                    ax.text(
                        0.05,
                        1.1,
                        f"t= {t:10f} ns",
                        transform=ax.transAxes,
                        fontsize=14,
                        verticalalignment="top",
                    )
                    plt.savefig(
                        f"./{filename}/cluster_frames/XY_cluster{cluster_index:03d}.png"
                    )
                    plt.close()

                cluster = [[clusterdata[i, 1], clusterdata[i, 2], clusterdata[i, 4]]]
                cluster_index += 1
    except Exception as e:
        print(e)


def plot_window(filename, t_window, verbose_level=1):
    """
    Function that plots time windows of raw pixel data.
    """
    print("Plotting frames")
    # load npy file
    pixdat = np.load(f"./{filename}/{filename}_pixeldata.npy")
    # find minimum and maximum time
    TmaxLine = max(pixdat, key=lambda x: x[3])
    TminLine = min(pixdat, key=lambda x: x[3])
    print(TmaxLine)
    print(TminLine)
    # make gif over time
    i = 0  # frame counter
    t_min = int(TminLine[3])
    t_max = int(TmaxLine[3])
    interval = int(t_window / 2)
    num_frames = int((t_max - t_min) / interval)
    print(f"generating {num_frames} frames")
    for t in range(t_min, t_max, interval):
        frame_pixels = []
        for pix in pixdat:
            if (t - t_window / 2) < pix[3] < (t + t_window / 2):
                frame_pixels.append(pix)

        grid = np.zeros((256, 256))
        for pix in frame_pixels:
            grid[int(pix[1]), int(pix[2])] = pix[4]

        fig = plt.figure(figsize=[10, 10], dpi=600)
        ax = fig.add_subplot(111)
        plt.imshow(grid)
        plt.colorbar()
        plt.clim(0, 10)
        props = ""
        ax.text(
            0.05,
            1.1,
            f"t= {t:10d} ns",
            transform=ax.transAxes,
            fontsize=14,
            verticalalignment="top",
        )
        plt.savefig(f"./{filename}/window_frames/XY_frame{i:03d}.png")
        plt.close()
        i += 1

## Mounting the measurement drive
In order to access the measurements, you will have to open the link shared with you by the TA with your RUG student account. The folder will then appear in your "Shared with me" tab in google drive. From there, copy over the files into a new folder, named 'TPX3 Data'.

Below is the code that mounts your google drive to the notebook. Running this will require you authorise Google colab to access your drive, after which all of the folders on your google drive will accessible, and you will be able to locate the folder with the TPX measurements under "/content/drive/MyDrive" in the file browser.

In [ ]:
import os
os.chdir('')  # Mac/Linux

## Processing the data
This is the code given to process the measurement data. With this, you will be able to convert the t3pa files into npy (numpy array), find and plot single- and multiple pixel clusters, and plot histograms of cluster energies. The notebook is structured in a typical workflow for this experiment. Feel free to edit and add to it however you see fit.



In [ ]:
# General things

verbose_level = 1       # Changes how much the code will 'talk to you'. Good for troubleshooting.

# Set the figure size and dpi
fig_size = (10, 5)
dpi = 600

### Converting the data
Before working with the data, it will need to be converted from its original format. The code also creates some directories that are needed for the other functions to work.

In [ ]:
# Converting the data is most likely something you want to do in batches.

filenames = [""]

for filename in filenames:
  # Create some directories for the output files. These directories are needed
  # for the code to work.

  if not os.path.isdir(filename):
      os.system(f"mkdir {filename}")
  if not os.path.isdir(f"./{filename}/window_frames"):
      os.system(f"mkdir ./{filename}/window_frames")
  if not os.path.isdir(f"./{filename}/cluster_frames"):
      os.system(f"mkdir ./{filename}/cluster_frames")
  
  convert_t3pa(filename)

## STEP 1 - Initial cluster finding (NPY file)


### Initial cluster finding and data plotting
To get an initial impression of the data, it is useful to plot general histograms of the sources. To do this, the code will need to look for clusters in order to then pick out the single-pixel clusters.

In [ ]:
# Finding clusters and plotting basic histograms (batched)

filenames = [""]

# For now you're dealing with uncalibrated data.
calibration_values = [1, 0, 0, 0]
linear_mapping = True

# Setting some limits for the histograms

y_limits = [0]            # If this is 0, there are no limits
x_ranges = [[0, 300]]     # 
numbox = 80               # You will want to adjust this when looking for specific peaks

for filename in filenames:
  # Find clusters: specifies the time window (ns), max cluster radius (pixels).
  cluster_finder(filename, 500, np.sqrt(2), verbose_level=verbose_level)
  # Save the cluster data, without calculating energies as this is unnecessary
  clusterdata_to_npy(filename, calibration_values, lin=linear_mapping, calc_e = False)
  # Plot single-pixel histogram
  plot_histograms(
      filename, num_box=numbox, ylim=y_limits, xrange=x_ranges, plt_e=False, single = True, plotlog = False
  )

## STEP 2 - Fit, finding peaks of ToT 

### Finding details in the spectra
Now you can move on to comparing the spectra of specific sources to ones found in literature. By adjusting the limits to the plots and the number of bins, you can reveal details and peaks in the spectra. Additionally, you can choose to do some curve fitting in order to get more accurate information on peaks and get standard deviations. Some 'examples' for limits are given below, so that you can get an idea of the syntax.

In [ ]:
# Finding more specific peaks in a spectrum and analysing the histogram

filename = ""

numbox = 200

# The code allows for multiple sets of limits, i.e. 'zooming' in to the data.
y_limits = [0]
x_ranges = x_ranges = [[0,300],[0.300]]
numbox = 200

# This plots the histograms and outputs the 'raw' data.
single_pixel_data = plot_histograms(
      filename, num_box=numbox, ylim=y_limits, xrange=x_ranges, plt_e=False, single = True, plotlog = False, return_data = True
)

# Converts the output data for curve fitting
tot_val, counts = np.histogram(single_pixel_data, numbox)

# Do your own curve fitting here : alrighty

import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

def gaussian(x, amp, mean, sigma):
    return amp * np.exp(-(x - mean)**2 / (2 * sigma**2)) 
# 2. Gaussian bins  
#                                                      v
counts, bin_edges = np.histogram(single_pixel_data, bins=150, range=(0, 300))
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

#  limits
tot_min, tot_max = [100, 160]
peak_mask = (bin_centers > tot_min) & (bin_centers < tot_max)

initial_guess = [max(counts[peak_mask]), (tot_min + tot_max) / 2, 5]
popt, pcov = curve_fit(gaussian, bin_centers[peak_mask], counts[peak_mask], p0=initial_guess)

peak_center = popt[1]
perr = np.sqrt(np.diag(pcov))
center_std_dev = perr[1] # std dvtion

print(f"Peak Center (ToT): {peak_center:.2f}")
print(f"Standard Deviation of Center: {center_std_dev:.4f}")

plt.figure(figsize=(10, 5))
plt.bar(bin_centers, counts, width=np.diff(bin_edges), alpha=0.5, color='gray', label='Data')
plt.plot(bin_centers[peak_mask], gaussian(bin_centers[peak_mask], *popt), color='red', lw=2, label='Gaussian Fit')
plt.xlim([0,300])
plt.ylim(0)
plt.xlabel("ToT Value")
plt.ylabel("Counts")
plt.legend()
plt.show()


## STEP 3 - Comparing with literature

### Calibration
Now you should implement your own code for calibration. Having found some characteristic peaks by comparing the data to spectra in literature, you can then use their ToT values and corresponding energies to obtain the fitting parameters $a$, $b$, $c$, and $t$ for the nonlinear expression:

$ \text{ToT}(E) = aE + b - \frac{c}{E-t}, $

where $a\neq 0$ and $\sqrt{a^2t^2 + 2abt + 4ac - 2at*\text{ToT} + b^2 - 2b*\text{ToT} + \text{ToT}^2} + at + b \neq 0$.

Some code is given to start you off, but feel free to program this however you see fit. Once you find values for the fitting parameters, you can use these in the data processing code above.

In [ ]:
energies = [31.0, 59.54] # Energies of the peaks identified
tot_vals = [73.5, 127.1] # ToT values of the peaks identified
tot_stds = [0.1759, 0.5738] # Add weights / standard deviation to the peaks identified

def energy_to_tot(E, a, b):
  return a * E + b

popt, pcov = curve_fit(energy_to_tot, energies, tot_vals, sigma=tot_stds, absolute_sigma=True)

a, b = popt
a_err, b_err = np.sqrt(np.diag(pcov))

calibration_values = [a, b]

print(f"Gain (a)   = {a:.4f} ± {a_err:.4f} ToT/keV")
print(f"Offset (b) = {b:.4f} ± {b_err:.4f} ToT\n")

E_range = np.linspace(10, 80, 200)
plt.figure(figsize=(7, 4.5))
plt.errorbar(energies, tot_vals, yerr=tot_stds, fmt='X', label='Experimental data points')
plt.plot(E_range, energy_to_tot(E_range, *popt), 
         label=f'Weighted Line: ToT = {a:.2f}*E + {b:.2f}')

plt.xlabel("Energy (keV)")
plt.ylabel("Time-over-Threshold (ToT)")
plt.grid(False)
plt.legend()
plt.show()
calibration_values = []

After this, you can use the code in order to convert all of the energies in your data.

In [ ]:
filenames = [""]

y_limits = [0]
x_ranges = [[0, 300]] # Opened up to 100 just in case
numbox = 80

for filename in filenames:
    # 1. This processes the raw file and generates a calibrated dataset on disk
    clusterdata_to_npy(filename, calibration_values, lin=True, calc_e=True)
    
    # 2. Tell the plotter to look at the calibrated file version!
    # Common framework suffix names: "_e", "_cal", or "_calibrated"
    calibrated_filename = filename 
    
    plot_histograms(
        calibrated_filename, num_box=numbox, ylim=y_limits, xrange=x_ranges, 
        plt_e=True, single=True, plotlog=False, return_data=False
    )

### Plotting clusters
The last excercise is to plot some multi-pixel clusters in order to identify examples coming from $\alpha$-, $\beta$-, and $\gamma$ radiation. In order to do this, the code will have to be set to look for some larger clusters. Once that is done, you can have it plot the clusters. Be aware that unless you stop it: the code will happily keep plotting all of the clusters it has found.

In [ ]:
# Find clusters and plot them

filename = ""

# Find clusters: play around with larger values for the time window and the 
# radius, be aware that this might take longer!

cluster_finder(filename, 1000, 10, verbose_level=verbose_level)
# Save the data
clusterdata_to_npy(filename, calibration_values, lin=False, calc_e = True)

min_clus_size = 2  # pixels. To define minimum cluster size to plot.
# Plot the clusters. This takes long!
plot_cluster(filename, min_clus_size, clim=20)